In [32]:
# Les importations
from datasets import load_dataset,Dataset, DatasetDict, concatenate_datasets


### Chargement du dataset

In [33]:
def dataset_loader(path) :
    with open(path, "r") as file:
        data = file.readlines()

    data_clean = [line for line in data if line.strip()]

    return data_clean


In [34]:
PATH = "../Data/full_dataset/"
data_arb_bible = dataset_loader(PATH + "arabe_tchad3.txt")
data_fr_bible = dataset_loader(PATH + "V_francais_3.txt")

## Ceux non biblique
data_arb_not_bible = dataset_loader(PATH + "arabic_not_bible")
data_fr_not_bible = dataset_loader(PATH + "french_not_bible")

In [35]:
# Création du dictionnaire
dataset_dict_bible = {"translation": [{"arb": a, "fr": f} for a, f in zip(data_arb_bible, data_fr_bible)]}
dataset_dict_not_bible = {"translation": [{"arb": a, "fr": f} for a, f in zip(data_arb_not_bible, data_fr_not_bible)]}


### Diviosion du dataset en train, test et validation

In [36]:
def equal_split(data_dict1 : dict, data_dict2 : None| dict) :
    # Création du dataset
    dataset1 = Dataset.from_dict(data_dict1)
    dataset2 = Dataset.from_dict(data_dict2)

    # The first dataset
    train_test_dataset1 = dataset1.train_test_split(test_size=0.2, seed= 42, shuffle= True)
    test_val_dataset1 = train_test_dataset1['test'].train_test_split(test_size= 0.5, seed= 42)

    # The second dataset
    train_test_dataset2 = dataset2.train_test_split(test_size=0.2, seed= 42, shuffle= True)
    test_val_dataset2 = train_test_dataset2['test'].train_test_split(test_size= 0.5, seed= 42)
    
    train_test_val_dataset = DatasetDict ({
        'train' : concatenate_datasets([train_test_dataset1['train'],train_test_dataset2['train']]).shuffle(seed = 42) ,
        'val'   : concatenate_datasets([test_val_dataset1['train'], test_val_dataset2['train']]).shuffle(seed = 42),
        'test'  : concatenate_datasets([test_val_dataset1['test'], test_val_dataset2['test']]).shuffle(42)
    })

    return train_test_val_dataset


In [37]:
train_test_val_dataset = equal_split(dataset_dict_bible, dataset_dict_not_bible)
train_test_val_dataset

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 30950
    })
    val: Dataset({
        features: ['translation'],
        num_rows: 3869
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 3869
    })
})

In [38]:
train_test_val_dataset.save_to_disk("../Data/train_test_split")

Saving the dataset (1/1 shards): 100%|██████████| 3869/3869 [00:00<00:00, 100614.20 examples/s]
